# DSC2026 — Prism-2B Private Inference — Global Length-Bucketed 2×T4

This version replaces **query-wise batching** with **global passage batching**.

Frozen scoring contract is unchanged:

- base: `infgrad/Prism-Qwen3.5-Reranker-2B`
- LoRA rank: inferred from checkpoint; expected `r=16`
- LoRA alpha: **32**
- passage aggregation: **top2_max**
- max length: **1024**
- score: `logit("yes") - logit("no")`

Optimization:

1. flatten every private passage globally;
2. tokenize once on CPU only to measure each truncated prompt length;
3. sort all passages by length (long → short);
4. alternate sorted records across both T4s for balanced work;
5. auto-benchmark batch sizes `16/32/64` on each GPU;
6. convert the fastest result into a dynamic **token budget**;
7. score neighboring lengths together, minimizing padding;
8. merge passage scores back with `max(passages)` for every `(qid, doc)`.

Attach a Kaggle Dataset containing:

- `best_state.pt`
- `PRISM_PRIVATE_PASSAGES.pkl`

Use **GPU T4 x2** and **Internet ON**.


## 1. Install dependencies

In [ ]:
import sys, subprocess

def pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip", *args]
    print("$", " ".join(cmd), flush=True)
    return subprocess.run(cmd, check=check)

# Old Kaggle torchao conflicts with current PEFT; Prism inference does not use it.
pip("uninstall", "-y", "torchao", check=False)

pip(
    "install", "-q", "--no-cache-dir", "-U",
    "transformers==5.17.0",
    "peft==0.21.0",
    "accelerate",
    "safetensors",
    "sentencepiece",
)

# Optional Qwen3.5 kernels. Failure is non-fatal; Transformers has reference
# PyTorch fallbacks, though those are slower.
for args in (
    ("install", "-q", "--no-build-isolation", "causal-conv1d"),
    ("install", "-q", "flash-linear-attention==0.5.2"),
):
    try:
        pip(*args)
    except subprocess.CalledProcessError:
        print("WARNING: optional kernel install failed:", args[-1], flush=True)

# Verify with a fresh interpreter, exactly as the workers will import it.
verify = r"""
import transformers, peft
from transformers import Qwen3_5TextConfig, Qwen3_5ForCausalLM
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("Qwen3.5 architecture: PASS")
try:
    import causal_conv1d
    print("causal_conv1d: PASS")
except Exception as e:
    print("causal_conv1d: unavailable:", type(e).__name__, e)
try:
    import fla
    print("flash-linear-attention/fla: PASS")
except Exception as e:
    print("flash-linear-attention/fla: unavailable:", type(e).__name__, e)
"""
subprocess.check_call([sys.executable, "-c", verify])


## 2. Verify GPUs and locate inputs

In [ ]:
import os, sys, pickle, json, hashlib
from pathlib import Path
import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name} | {p.total_memory/2**30:.1f} GiB")

assert torch.cuda.device_count() >= 2, "Select Kaggle accelerator: GPU T4 x2"

INPUT = Path("/kaggle/input")
ck = list(INPUT.rglob("best_state.pt"))
wl = list(INPUT.rglob("PRISM_PRIVATE_PASSAGES.pkl"))

print("checkpoint hits:", ck)
print("workload hits:", wl)

assert len(ck) == 1, f"Need exactly one best_state.pt, got {len(ck)}"
assert len(wl) == 1, f"Need exactly one PRISM_PRIVATE_PASSAGES.pkl, got {len(wl)}"

CHECKPOINT = ck[0]
WORKLOAD = wl[0]

OUT = Path("/kaggle/working/prism_private_bucketed")
OUT.mkdir(parents=True, exist_ok=True)

print("CHECKPOINT:", CHECKPOINT)
print("WORKLOAD:", WORKLOAD)
print("OUT:", OUT)


## 3. Build the global length manifest (CPU)

This does **not** run Prism. It only tokenizes prompts to obtain their truncated
lengths. The resulting manifest is small: one tuple per passage.

A record is:

`(record_id, qid, doc_id, passage_index, token_length)`

The manifest is deterministic and reused if its workload hash matches.


In [ ]:
import pickle, hashlib, time
from pathlib import Path
from transformers import AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()

BASE_MODEL = "infgrad/Prism-Qwen3.5-Reranker-2B"
MAX_LENGTH = 1024

SYSTEM_PROMPT = (
    "Judge whether the Document meets the requirements based on "
    "the Query and the Instruct provided. "
)
INSTRUCTION = (
    'Judge if the document is relevant to the query. Reply "yes" or "no".\n'
    'On "yes", also emit:\n'
    "<contribution>One sentence covering every core point the document "
    "contributes to the query, without elaboration.</contribution>\n"
    "<evidence>Self-contained rewrite of the query-relevant content. Rules:\n"
    "- Faithful: rephrase only; add or infer nothing.\n"
    "- Self-contained: evidence alone must fully answer the query.\n"
    "- Concise: drop query-irrelevant background.\n"
    "- Verbatim (no translation): proper nouns, terms, abbreviations, "
    "numbers, dates, code, URLs.\n"
    "- Output language: multilingual doc → query's language; else doc's language."
    "</evidence>"
)
TEMPLATE = (
    "<|im_start|>system\n{system}<|im_end|>\n"
    "<|im_start|>user\n"
    "<Instruct>: {instruction}\n"
    "<Query>: {query}\n"
    "<Document>: {doc}<|im_end|>\n"
    "<|im_start|>assistant\n<think>\n\n</think>\n\n"
)

def mkprompt(q, d):
    return TEMPLATE.format(
        system=SYSTEM_PROMPT,
        instruction=INSTRUCTION,
        query=q,
        doc=d,
    )

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_path = OUT / "PRISM_GLOBAL_LENGTH_MANIFEST.pkl"
workload_sha = sha256_file(WORKLOAD)

reuse = False
if manifest_path.exists():
    old_manifest = pickle.loads(manifest_path.read_bytes())
    reuse = (
        old_manifest.get("workload_sha256") == workload_sha
        and old_manifest.get("max_length") == MAX_LENGTH
        and old_manifest.get("base_model") == BASE_MODEL
    )

if reuse:
    manifest = old_manifest
    print("Reusing existing manifest:", manifest_path)
else:
    obj = pickle.loads(WORKLOAD.read_bytes())
    queries = obj["queries"]
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)

    compact = []
    for qid in sorted(queries):
        row = queries[qid]
        for doc_id in row["docs"]:
            for passage_idx, passage in enumerate(row["docs"][doc_id]):
                compact.append(
                    (str(qid), str(doc_id), int(passage_idx), row["question"], passage)
                )

    print(f"Passages to length-scan: {len(compact):,}")
    records = []
    chunk_size = 512
    t0 = time.perf_counter()

    for start in range(0, len(compact), chunk_size):
        chunk = compact[start:start + chunk_size]
        prompts = [mkprompt(x[3], x[4]) for x in chunk]

        enc = tok(
            prompts,
            add_special_tokens=False,
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
            return_length=True,
            return_attention_mask=False,
        )
        lengths = list(map(int, enc["length"]))

        for j, ((qid, doc_id, passage_idx, _, _), length) in enumerate(
            zip(chunk, lengths)
        ):
            rid = start + j
            records.append((rid, qid, doc_id, passage_idx, length))

        if start == 0 or (start // chunk_size) % 25 == 0:
            elapsed = time.perf_counter() - t0
            done = min(start + len(chunk), len(compact))
            print(
                f"length scan {done:,}/{len(compact):,} "
                f"({100*done/len(compact):.1f}%) | {elapsed:.1f}s",
                flush=True,
            )

    # Long -> short: within any dynamic batch, the first element is the max
    # length, so token-budget batching has an exact upper bound.
    records.sort(key=lambda r: r[4], reverse=True)

    manifest = {
        "schema": "manual.prism_global_length_manifest.v3",
        "base_model": BASE_MODEL,
        "max_length": MAX_LENGTH,
        "workload_sha256": workload_sha,
        "records": records,
    }
    manifest_path.write_bytes(pickle.dumps(manifest, protocol=5))

lengths = [r[4] for r in manifest["records"]]
lengths_sorted = sorted(lengths)

def pct(p):
    if not lengths_sorted:
        return None
    idx = round((len(lengths_sorted)-1) * p)
    return lengths_sorted[idx]

print("="*88)
print("MANIFEST READY:", manifest_path)
print("passages:", f"{len(lengths):,}")
print(
    "lengths:",
    "min", min(lengths),
    "| p25", pct(.25),
    "| p50", pct(.50),
    "| p75", pct(.75),
    "| p90", pct(.90),
    "| p95", pct(.95),
    "| max", max(lengths),
)
print(
    "truncated@1024:",
    f"{sum(x >= 1024 for x in lengths):,}",
    f"({100*sum(x >= 1024 for x in lengths)/len(lengths):.2f}%)",
)
print("="*88)


## 4. Write the optimized GPU worker

In [ ]:
from pathlib import Path

WORKER = '#!/usr/bin/env python\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport pickle\nimport time\nfrom pathlib import Path\n\n# Quiet HF before importing transformers.\nos.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")\nos.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")\nos.environ.setdefault("TRANSFORMERS_NO_ADVISORY_WARNINGS", "1")\nos.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n\nimport numpy as np\nimport torch\n\nBASE_MODEL = "infgrad/Prism-Qwen3.5-Reranker-2B"\n\nSYSTEM_PROMPT = (\n    "Judge whether the Document meets the requirements based on "\n    "the Query and the Instruct provided. "\n)\nINSTRUCTION = (\n    \'Judge if the document is relevant to the query. Reply "yes" or "no".\\n\'\n    \'On "yes", also emit:\\n\'\n    "<contribution>One sentence covering every core point the document "\n    "contributes to the query, without elaboration.</contribution>\\n"\n    "<evidence>Self-contained rewrite of the query-relevant content. Rules:\\n"\n    "- Faithful: rephrase only; add or infer nothing.\\n"\n    "- Self-contained: evidence alone must fully answer the query.\\n"\n    "- Concise: drop query-irrelevant background.\\n"\n    "- Verbatim (no translation): proper nouns, terms, abbreviations, "\n    "numbers, dates, code, URLs.\\n"\n    "- Output language: multilingual doc → query\'s language; else doc\'s language."\n    "</evidence>"\n)\nTEMPLATE = (\n    "<|im_start|>system\\n{system}<|im_end|>\\n"\n    "<|im_start|>user\\n"\n    "<Instruct>: {instruction}\\n"\n    "<Query>: {query}\\n"\n    "<Document>: {doc}<|im_end|>\\n"\n    "<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n)\n\ndef mkprompt(question: str, passage: str) -> str:\n    return TEMPLATE.format(\n        system=SYSTEM_PROMPT,\n        instruction=INSTRUCTION,\n        query=question,\n        doc=passage,\n    )\n\ndef atomic_pickle(path: Path, obj) -> None:\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_bytes(pickle.dumps(obj, protocol=5))\n    os.replace(tmp, path)\n\ndef atomic_json(path: Path, obj) -> None:\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    tmp.write_text(json.dumps(obj, indent=2), encoding="utf-8")\n    os.replace(tmp, path)\n\ndef infer_lora_contract(state):\n    a_keys = [k for k in state if ".lora_A." in k]\n    if not a_keys:\n        raise RuntimeError("No LoRA A tensors found")\n    ranks = {int(state[k].shape[0]) for k in a_keys}\n    if len(ranks) != 1:\n        raise RuntimeError(f"Mixed LoRA ranks: {ranks}")\n    rank = next(iter(ranks))\n    targets = sorted({\n        k.split(".lora_A.")[0].split(".")[-1]\n        for k in a_keys\n    })\n    return rank, targets\n\ndef load_model(checkpoint: Path, gpu: int):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from transformers.utils import logging as hf_logging\n    from peft import LoraConfig, TaskType, get_peft_model\n\n    hf_logging.disable_progress_bar()\n\n    torch.cuda.set_device(gpu)\n    device = f"cuda:{gpu}"\n\n    print(f"loading checkpoint: {checkpoint}", flush=True)\n    obj = torch.load(checkpoint, map_location="cpu", weights_only=True)\n    state = obj.get("state_dict", obj)\n    rank, targets = infer_lora_contract(state)\n\n    print(\n        f"LoRA rank={rank} alpha=32 tensors={len(state)} "\n        f"targets={\',\'.join(targets)}",\n        flush=True,\n    )\n\n    tok = AutoTokenizer.from_pretrained(BASE_MODEL)\n    tok.padding_side = "left"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n\n    print(f"loading base: {BASE_MODEL}", flush=True)\n    base = AutoModelForCausalLM.from_pretrained(\n        BASE_MODEL,\n        dtype=torch.float16,\n        low_cpu_mem_usage=True,\n        attn_implementation="sdpa",\n    )\n\n    cfg = LoraConfig(\n        r=rank,\n        lora_alpha=32,\n        target_modules=targets,\n        lora_dropout=0.0,\n        bias="none",\n        task_type=TaskType.CAUSAL_LM,\n    )\n    model = get_peft_model(base, cfg)\n    incompat = model.load_state_dict(state, strict=False)\n    if incompat.unexpected_keys:\n        raise RuntimeError(\n            f"Unexpected checkpoint keys: {incompat.unexpected_keys[:10]}"\n        )\n\n    model.eval().to(device)\n\n    yes_ids = tok.encode("yes", add_special_tokens=False)\n    no_ids = tok.encode("no", add_special_tokens=False)\n    if len(yes_ids) != 1 or len(no_ids) != 1:\n        raise RuntimeError(\n            f"Unexpected yes/no tokenization: yes={yes_ids}, no={no_ids}"\n        )\n\n    alloc = torch.cuda.memory_allocated(gpu) / 2**30\n    reserved = torch.cuda.memory_reserved(gpu) / 2**30\n    print(\n        f"model ready | {device} | allocated={alloc:.2f} GiB "\n        f"reserved={reserved:.2f} GiB",\n        flush=True,\n    )\n    return model, tok, yes_ids[0], no_ids[0], device\n\n@torch.inference_mode()\ndef score_texts(\n    model,\n    tok,\n    yes_id: int,\n    no_id: int,\n    device: str,\n    texts: list[str],\n    max_length: int,\n):\n    enc = tok(\n        texts,\n        padding=True,\n        truncation=True,\n        max_length=max_length,\n        return_tensors="pt",\n        add_special_tokens=False,\n    )\n    enc = {\n        k: v.to(device, non_blocking=True)\n        for k, v in enc.items()\n    }\n\n    # The PEFT-injected LoRA layers remain inside get_base_model().\n    base = model.get_base_model()\n    with torch.autocast(device_type="cuda", dtype=torch.float16):\n        out = base.model(\n            input_ids=enc["input_ids"],\n            attention_mask=enc.get("attention_mask"),\n            use_cache=False,\n            return_dict=True,\n        )\n        last = out.last_hidden_state[:, -1, :]\n        logits = base.lm_head(last).float()\n        score = logits[:, yes_id] - logits[:, no_id]\n\n    return score.detach().cpu().tolist()\n\ndef build_prompts(records, queries):\n    prompts = []\n    for rid, qid, doc_id, passage_idx, tok_len in records:\n        row = queries[qid]\n        passage = row["docs"][doc_id][passage_idx]\n        prompts.append(mkprompt(row["question"], passage))\n    return prompts\n\ndef benchmark_batch_sizes(\n    model,\n    tok,\n    yes_id,\n    no_id,\n    device,\n    records,\n    queries,\n    max_length,\n    candidates,\n    sample_n,\n):\n    if not records:\n        raise RuntimeError("No records for autotune")\n\n    # Records are sorted long->short. Choose a representative p75-length window:\n    # in descending order, p75 lies about 25% into the array.\n    n = min(sample_n, len(records))\n    center = max(0, min(len(records) - n, len(records) // 4 - n // 2))\n    sample = records[center:center + n]\n    prompts = build_prompts(sample, queries)\n    lengths = [r[4] for r in sample]\n    rep_len = max(lengths)\n\n    print(\n        f"AUTOTUNE sample={len(sample)} token_len="\n        f"{min(lengths)}..{max(lengths)} median={int(np.median(lengths))}",\n        flush=True,\n    )\n\n    # Small warm-up to exclude lazy CUDA/kernel init from benchmark timings.\n    warm = prompts[: min(4, len(prompts))]\n    _ = score_texts(\n        model, tok, yes_id, no_id, device, warm, max_length\n    )\n    torch.cuda.synchronize()\n\n    rows = []\n    for bs in candidates:\n        bs = int(bs)\n        if bs <= 0:\n            continue\n        torch.cuda.empty_cache()\n        torch.cuda.reset_peak_memory_stats()\n        torch.cuda.synchronize()\n        t0 = time.perf_counter()\n        ok = True\n        error = None\n\n        try:\n            for i in range(0, len(prompts), bs):\n                _ = score_texts(\n                    model,\n                    tok,\n                    yes_id,\n                    no_id,\n                    device,\n                    prompts[i:i + bs],\n                    max_length,\n                )\n            torch.cuda.synchronize()\n        except torch.cuda.OutOfMemoryError:\n            ok = False\n            error = "OOM"\n            torch.cuda.empty_cache()\n\n        dt = time.perf_counter() - t0\n        peak_gib = torch.cuda.max_memory_allocated() / 2**30\n\n        if ok:\n            pps = len(prompts) / dt\n            rows.append({\n                "batch_size": bs,\n                "seconds": dt,\n                "prompts_per_second": pps,\n                "peak_vram_gib": peak_gib,\n            })\n            print(\n                f"AUTOTUNE bs={bs:>3} | {dt:7.2f}s | "\n                f"{pps:6.2f} prompts/s | peak={peak_gib:.2f} GiB",\n                flush=True,\n            )\n        else:\n            print(\n                f"AUTOTUNE bs={bs:>3} | OOM | peak={peak_gib:.2f} GiB",\n                flush=True,\n            )\n\n    if not rows:\n        raise RuntimeError("All autotune batch sizes OOM")\n\n    best = max(rows, key=lambda x: x["prompts_per_second"])\n    best_bs = int(best["batch_size"])\n\n    # Convert the measured best batch at representative length into a token\n    # budget. For shorter records, batches grow (capped separately); for\n    # longer records, batches shrink.\n    token_budget = best_bs * rep_len\n\n    print(\n        f"AUTOTUNE BEST bs={best_bs} at rep_len={rep_len} "\n        f"=> token_budget={token_budget:,}",\n        flush=True,\n    )\n    return token_budget, rows\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--gpu", type=int, required=True)\n    ap.add_argument("--shard", type=int, required=True)\n    ap.add_argument("--num-shards", type=int, default=2)\n    ap.add_argument("--workload", type=Path, required=True)\n    ap.add_argument("--manifest", type=Path, required=True)\n    ap.add_argument("--checkpoint", type=Path, required=True)\n    ap.add_argument("--output", type=Path, required=True)\n    ap.add_argument("--report", type=Path, required=True)\n    ap.add_argument("--max-length", type=int, default=1024)\n    ap.add_argument("--max-batch", type=int, default=128)\n    ap.add_argument("--autotune-sample", type=int, default=64)\n    ap.add_argument(\n        "--autotune-batches",\n        type=str,\n        default="16,32,64",\n    )\n    ap.add_argument("--save-every-batches", type=int, default=20)\n    ap.add_argument("--log-every-batches", type=int, default=5)\n    args = ap.parse_args()\n\n    print(\n        f"worker start | gpu={args.gpu} shard={args.shard}/"\n        f"{args.num_shards} max_batch={args.max_batch}",\n        flush=True,\n    )\n\n    model, tok, yes_id, no_id, device = load_model(\n        args.checkpoint, args.gpu\n    )\n\n    workload_obj = pickle.loads(args.workload.read_bytes())\n    queries = workload_obj["queries"]\n    manifest = pickle.loads(args.manifest.read_bytes())\n\n    if manifest["max_length"] != args.max_length:\n        raise RuntimeError(\n            f"Manifest max_length={manifest[\'max_length\']} "\n            f"!= worker max_length={args.max_length}"\n        )\n\n    all_records = manifest["records"]\n\n    # Global records were sorted by token length descending. Alternating them\n    # across GPUs makes both shards see almost the same length distribution and\n    # balances compute substantially better than qid-based sharding.\n    records = [\n        rec\n        for i, rec in enumerate(all_records)\n        if i % args.num_shards == args.shard\n    ]\n\n    saved = (\n        pickle.loads(args.output.read_bytes())\n        if args.output.is_file()\n        else {}\n    )\n    # Normalize possible int/string keys from previous run.\n    saved = {int(k): float(v) for k, v in saved.items()}\n\n    remaining = [rec for rec in records if int(rec[0]) not in saved]\n    remaining.sort(key=lambda r: r[4], reverse=True)\n\n    print(\n        f"records | shard={len(records):,} cached={len(saved):,} "\n        f"remaining={len(remaining):,}",\n        flush=True,\n    )\n\n    if not remaining:\n        print("shard already complete", flush=True)\n        return\n\n    candidates = [\n        int(x) for x in args.autotune_batches.split(",") if x.strip()\n    ]\n    token_budget, bench = benchmark_batch_sizes(\n        model=model,\n        tok=tok,\n        yes_id=yes_id,\n        no_id=no_id,\n        device=device,\n        records=remaining,\n        queries=queries,\n        max_length=args.max_length,\n        candidates=candidates,\n        sample_n=args.autotune_sample,\n    )\n\n    # Cap the dynamic budget so accidental odd-length benchmark samples cannot\n    # produce a pathological giant batch.\n    token_budget = max(4096, int(token_budget))\n\n    total = len(remaining)\n    done = 0\n    batch_no = 0\n    start = time.perf_counter()\n    padded_tokens_done = 0\n    raw_tokens_done = 0\n\n    i = 0\n    current_max_batch = int(args.max_batch)\n\n    while i < total:\n        max_len = max(1, int(remaining[i][4]))\n        bs = min(\n            current_max_batch,\n            max(1, token_budget // max_len),\n            total - i,\n        )\n        batch = remaining[i:i + bs]\n        prompts = build_prompts(batch, queries)\n\n        try:\n            scores = score_texts(\n                model,\n                tok,\n                yes_id,\n                no_id,\n                device,\n                prompts,\n                args.max_length,\n            )\n        except torch.cuda.OutOfMemoryError:\n            torch.cuda.empty_cache()\n            if bs <= 1:\n                raise\n            # Reduce global max batch so future shorter batches are also safer.\n            new_cap = max(1, bs // 2)\n            current_max_batch = min(current_max_batch, new_cap)\n            token_budget = min(token_budget, new_cap * max_len)\n            print(\n                f"CUDA OOM | len={max_len} bs={bs} -> "\n                f"max_batch={current_max_batch} "\n                f"token_budget={token_budget:,}; retry",\n                flush=True,\n            )\n            continue\n\n        if len(scores) != len(batch):\n            raise RuntimeError(\n                f"score count mismatch {len(scores)} != {len(batch)}"\n            )\n\n        for rec, score in zip(batch, scores):\n            rid = int(rec[0])\n            if not np.isfinite(score):\n                raise RuntimeError(\n                    f"non-finite score rid={rid}: {score}"\n                )\n            saved[rid] = float(score)\n\n        raw_tokens = sum(int(r[4]) for r in batch)\n        padded_tokens = max_len * len(batch)\n        raw_tokens_done += raw_tokens\n        padded_tokens_done += padded_tokens\n\n        i += len(batch)\n        done += len(batch)\n        batch_no += 1\n\n        if batch_no % args.save_every_batches == 0:\n            atomic_pickle(args.output, saved)\n\n        if (\n            batch_no == 1\n            or batch_no % args.log_every_batches == 0\n            or done == total\n        ):\n            elapsed = time.perf_counter() - start\n            rps = done / elapsed\n            eta_min = (total - done) / max(rps, 1e-9) / 60\n            pad_eff = raw_tokens_done / max(padded_tokens_done, 1)\n            peak = torch.cuda.max_memory_allocated() / 2**30\n            print(\n                f"progress {done:,}/{total:,} ({100*done/total:5.1f}%) | "\n                f"{rps:6.2f} passages/s | ETA {eta_min:6.1f}m | "\n                f"len={max_len:4d} bs={len(batch):3d} | "\n                f"pad_eff={100*pad_eff:5.1f}% | peak={peak:.2f} GiB",\n                flush=True,\n            )\n\n    atomic_pickle(args.output, saved)\n\n    elapsed = time.perf_counter() - start\n    report = {\n        "gpu": args.gpu,\n        "shard": args.shard,\n        "records": len(records),\n        "elapsed_seconds_scoring": elapsed,\n        "passages_per_second": total / elapsed,\n        "token_budget": token_budget,\n        "max_batch_final": current_max_batch,\n        "padding_efficiency": raw_tokens_done / max(padded_tokens_done, 1),\n        "autotune": bench,\n        "output": str(args.output),\n    }\n    atomic_json(args.report, report)\n\n    print(\n        f"COMPLETE | records={len(saved):,}/{len(records):,} | "\n        f"score_time={elapsed/60:.1f}m | "\n        f"{total/elapsed:.2f} passages/s | "\n        f"pad_eff={100*report[\'padding_efficiency\']:.1f}%",\n        flush=True,\n    )\n\nif __name__ == "__main__":\n    main()\n'

WORKER_PATH = Path('/kaggle/working/prism_bucket_worker_v3.py')
WORKER_PATH.write_text(WORKER, encoding='utf-8')
print('worker:', WORKER_PATH)
print('bytes:', WORKER_PATH.stat().st_size)


## 5. Run both T4s — live terminal output

Each GPU first benchmarks batch sizes **16 / 32 / 64** on a representative
64-passage length window. It converts the fastest batch into a token budget.

Then global sorted records are processed with approximately:

`batch_size = min(128, token_budget // longest_sequence_in_batch)`

So short passages can use large batches, while 1024-token passages
automatically use smaller batches.

The two workers stream logs live as `[GPU0]` and `[GPU1]`.

Partial **passage-level** caches are resumable.


In [ ]:
import os, sys, time, queue, threading, subprocess

# Stop stale workers from an earlier attempt only.
for pattern in (
    "/kaggle/working/prism_worker.py",
    "/kaggle/working/prism_bucket_worker_v3.py",
):
    subprocess.run(
        ["pkill", "-f", pattern],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
time.sleep(1)

env = os.environ.copy()
env["TOKENIZERS_PARALLELISM"] = "false"
env["PYTHONUNBUFFERED"] = "1"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TRANSFORMERS_VERBOSITY"] = "error"
env["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

procs = []
events = queue.Queue()

def pump(gpu, proc):
    try:
        for line in proc.stdout:
            events.put((gpu, line.rstrip("\n")))
    finally:
        events.put((gpu, None))

for gpu in (0, 1):
    cmd = [
        sys.executable,
        str(WORKER_PATH),
        "--gpu", str(gpu),
        "--shard", str(gpu),
        "--num-shards", "2",
        "--workload", str(WORKLOAD),
        "--manifest", str(manifest_path),
        "--checkpoint", str(CHECKPOINT),
        "--output", str(OUT / f"passage_scores_gpu{gpu}.pkl"),
        "--report", str(OUT / f"worker_report_gpu{gpu}.json"),
        "--max-length", "1024",
        "--max-batch", "128",
        "--autotune-sample", "64",
        "--autotune-batches", "16,32,64",
        "--save-every-batches", "20",
        "--log-every-batches", "5",
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    procs.append(proc)
    threading.Thread(
        target=pump,
        args=(gpu, proc),
        daemon=True,
    ).start()
    print(f"[MAIN] launched GPU{gpu} pid={proc.pid}", flush=True)

finished = 0
try:
    while finished < 2:
        try:
            gpu, line = events.get(timeout=1.0)
        except queue.Empty:
            continue

        if line is None:
            finished += 1
        else:
            print(f"[GPU{gpu}] {line}", flush=True)

except KeyboardInterrupt:
    print("\n[MAIN] interrupted — terminating workers...", flush=True)
    for proc in procs:
        if proc.poll() is None:
            proc.terminate()
    raise

codes = [p.wait() for p in procs]
print("[MAIN] return codes:", codes)
if codes != [0, 0]:
    raise RuntimeError(f"Worker failure: {codes}")

print("[MAIN] BOTH GPU SHARDS COMPLETE")


## 6. Merge passage scores → document `top2_max` and validate

This restores the exact output format expected by
`materialize_private_prism_d1_v1.py`:

`{qid: {doc_id: prism_score}}`


In [ ]:
import pickle, json, hashlib, math
from pathlib import Path

workload_obj = pickle.loads(WORKLOAD.read_bytes())
queries = workload_obj["queries"]
manifest = pickle.loads(manifest_path.read_bytes())
records = manifest["records"]

passage_scores = {}
for gpu in (0, 1):
    p = OUT / f"passage_scores_gpu{gpu}.pkl"
    assert p.is_file(), f"Missing worker cache: {p}"
    part = pickle.loads(p.read_bytes())
    part = {int(k): float(v) for k, v in part.items()}

    overlap = set(passage_scores) & set(part)
    assert not overlap, f"GPU passage-id overlap: {list(overlap)[:10]}"
    passage_scores.update(part)

expected_rids = {int(r[0]) for r in records}
missing_rids = expected_rids - set(passage_scores)
extra_rids = set(passage_scores) - expected_rids

assert not missing_rids, f"Missing passage scores: {list(missing_rids)[:20]}"
assert not extra_rids, f"Unexpected passage scores: {list(extra_rids)[:20]}"

merged = {}
for rid, qid, doc_id, passage_idx, token_len in records:
    score = float(passage_scores[int(rid)])
    qrow = merged.setdefault(str(qid), {})
    doc_id = str(doc_id)
    if doc_id not in qrow or score > qrow[doc_id]:
        qrow[doc_id] = score

# Exact workload coverage check.
missing_docs = []
extra_docs = []
pair_count = 0

for qid, row in queries.items():
    expected_docs = set(map(str, row["docs"]))
    got_docs = set(merged.get(str(qid), {}))
    pair_count += len(expected_docs)

    for d in expected_docs - got_docs:
        missing_docs.append((str(qid), d))
    for d in got_docs - expected_docs:
        extra_docs.append((str(qid), d))

assert len(merged) == 2080, f"Expected 2080 qids, got {len(merged)}"
assert not missing_docs, f"Missing docs: {missing_docs[:20]}"
assert not extra_docs, f"Extra docs: {extra_docs[:20]}"

final = Path("/kaggle/working/prism_private_scores.pkl")
final.write_bytes(pickle.dumps(merged, protocol=5))

sha = hashlib.sha256(final.read_bytes()).hexdigest()
doc_vals = [s for row in merged.values() for s in row.values()]
passage_vals = list(passage_scores.values())

worker_reports = {}
for gpu in (0, 1):
    rp = OUT / f"worker_report_gpu{gpu}.json"
    if rp.exists():
        worker_reports[str(gpu)] = json.loads(rp.read_text())

report = {
    "schema": "manual.kaggle_prism_global_bucketed_2xt4.v3",
    "queries": len(merged),
    "document_pairs": pair_count,
    "passages": len(passage_scores),
    "doc_score_min": float(min(doc_vals)),
    "doc_score_max": float(max(doc_vals)),
    "passage_score_min": float(min(passage_vals)),
    "passage_score_max": float(max(passage_vals)),
    "sha256": sha,
    "base": "infgrad/Prism-Qwen3.5-Reranker-2B",
    "alpha": 32,
    "mode": "top2_max",
    "max_length": 1024,
    "workers": worker_reports,
}

report_path = Path("/kaggle/working/PRISM_KAGGLE_BUCKETED_REPORT.json")
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("="*88)
print("VALIDATION PASS")
print("queries:", len(merged))
print("document pairs:", f"{pair_count:,}")
print("passages:", f"{len(passage_scores):,}")
print("sha256:", sha)
print("FINAL:", final)
print("REPORT:", report_path)
print("="*88)


## 7. Optional: inspect worker speed reports

The most useful fields are:

- `passages_per_second`
- `padding_efficiency`
- `autotune`
- `token_budget`

For comparison, the old query-wise runner was around **34–37 s/query**
on the first 15 queries. This notebook reports passage throughput directly,
which is the meaningful unit after flattening.


In [ ]:
import json
for gpu in (0, 1):
    p = OUT / f"worker_report_gpu{gpu}.json"
    if p.exists():
        print("="*88)
        print("GPU", gpu)
        print(json.dumps(json.loads(p.read_text()), indent=2))
